In [1]:
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
pd.options.mode.chained_assignment = None 
pd.options.display.max_colwidth = 100
pd.set_option('display.width', 1500)
pd.set_option('display.max_colwidth', 1000)

/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.18) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


In [2]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
   

    df = pd.DataFrame(tracker.get_all_records(head=3))

    return df
            
df = getXrefs()

In [3]:
display(df.loc[(df['State Steward'] == 'CDOS') & (df['Category'] == 'Lobbyist'),
        ['Dataset Title','Short Description','Category','Type']])

,Dataset Title,Short Description,Category,Type
11,Bill Information and Position with Income of Lobbyist in Colorado,Information for each lobbyist and their associated client with the reported bills and positions associated with the client and the total gross income the client paid to lobbyist for the report month for the State of Colorado provided by the Colorado Department of State (CDOS).,Lobbyist,Bills
133,Characterization of Lobbyist Clients in Colorado,"Information for each lobbyist, including contact details, and their associated client and contact details, client business description and the name of the CEO provided by the Colorado Department of State (CDOS).",Lobbyist,Client
191,Directory of Lobbyist Clients in Colorado,Lobbyist name and address and the names and addresses of their associated clients provided by the Colorado Department of State (CDOS).,Lobbyist,Client
192,Directory of Lobbyists in Colorado,"Information for each registered lobbyist, including contact details, and their associated income and expenses as summarized by month and associated report date for the State of Colorado dating back to 1995 provided by the Colorado Department of State (CDOS).",Lobbyist,Lobbyist
205,Expenses for Lobbyists in Colorado,Registered lobbyist expenses for the State of Colorado dating back to 1995 provided by the Colorado Department of State (CDOS).,Lobbyist,Bills
358,Subcontractors for Lobbyists in Colorado,Information for each lobbyist that hires another lobbyist or lobbying business (subcontractor) along with total gross income paid ito the subcontractor in the given report month for the State of Colorado dating back to 2002 provided by the Colorado Department of State (CDOS).,Lobbyist,Lobbyist


In [4]:
files

NameError: name 'files' is not defined

In [36]:
fin=open("list-new")
files= fin.readlines()
dfsNew = {}
new_columns={}
for file in files:
    file=file.strip()
    df=pd.read_csv(f"/home/joe/bic_etl/cdos/lobbyist/data_source/new_format/{file}",delimiter="\t",encoding="latin")
    dfsNew[file]=df
    new_columns[file]={}
    new_columns[file]['columns']=set(df.columns)
    new_columns[file]['nrows']=df.shape[0]
    
  #  print(df.columns)


In [38]:
fin=open("list-old")
files= fin.readlines()
old_columns_all = set()
old_columns={}
dfsOld = {}
for file in files:
 
    file=file.strip()
    df=pd.read_csv(f"/home/joe/bic_etl/cdos/lobbyist/data_source/{file}",delimiter="\t",encoding="latin")

    dfsOld[file]=df
    old_columns[file]={}
    old_columns[file]['columns']=set(df.columns)
    old_columns[file]['nrows']=df.shape[0]
    
    
    
    if len(old_columns_all) > 0:
        old_columns_all=old_columns_all.union(set(df.columns))
    else:
        old_columns_all=set(df.columns)
#    print(df.columns)

/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (10,11,12,13,15,16,17,18,19,20,24,25,26,27,28,30) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [29]:
len(old_columns_all)

57

In [39]:
for file in old_columns.keys():
    columnsO = old_columns[file]['columns']
    nrowsO = old_columns[file]['nrows']
    for file_new in new_columns.keys():
         columnsN = new_columns[file_new]['columns']
         nrowsN = new_columns[file_new]['nrows']
        
         a=columnsO-columnsN
         b=columnsN-columnsO
        
         print(f"{nrowsO:6.0f}  {nrowsN:6.0f}  {len(columnsO):2.0f}  {len(columnsN):2.0f} {len(a):2.0f}  {len(b):2.0f} {file:25s} {file_new:25s} ")
#          print(file,file_new)

 87995   35570  15  16  9  10 Client.txt                prof_bills.txt            
 87995    5349  15  21  6  12 Client.txt                prof_clients.txt          
 87995    5081  15  16 11  12 Client.txt                prof_disclosure_report_summary.txt 
 87995     329  15  16 10  11 Client.txt                prof_expenditures.txt     
 87995    9299  15  17  9  11 Client.txt                prof_income.txt           
 87995    4967  15  18 10  13 Client.txt                prof_lobbyist_directory.txt 
 87995     510  15  17 11  13 Client.txt                prof_subcontractors.txt   
 87995   42872  15  13 12  10 Client.txt                state_lobbyist_bills.txt  
 87995     225  15  17 12  14 Client.txt                state_lobbyist_directory.txt 
 87995    7993  15  13 10   8 Client.txt                state_lobbyist_funds_hours.txt 
 87995    6219  15  16 10  11 Client.txt                state_lobbyist_officials.txt 
196254   35570  18  16 11   9 SummaryInfo.txt           prof_bill

In [ ]:
87995    5349  15  21  6  12 Client.txt                prof_clients.txt


In [11]:
old_columns["SummaryInfo.txt"]
new_columns["prof_lobbyist_directory.txt"]

{'fiscalYearsRegistered',
 'lobbyistAddress',
 'lobbyistCity',
 'lobbyistDesignation',
 'lobbyistFirmName',
 'lobbyistFirstName',
 'lobbyistMiddleName',
 'lobbyistName',
 'lobbyistPhone',
 'lobbyistState',
 'lobbyistSuffix',
 'lobbyistType',
 'lobbyistZip',
 'lobbystLastName',
 'primaryLobbyistID',
 'registrationEndDate',
 'runDate',
 'status'}

In [12]:
old_columns["SummaryInfo.txt"]


{'annualLobbyistRegistrationID',
 'businessAssociatedWithPendingLegislation',
 'dateDisclosureFiled',
 'dateDisclosureLastModified',
 'fiscalYear',
 'lobbyistAddress1',
 'lobbyistAddress2',
 'lobbyistCity',
 'lobbyistFirstName',
 'lobbyistLastName',
 'lobbyistState',
 'lobbyistZip',
 'lobbyistZipofficialStateLobbyist',
 'primaryLobbyistID',
 'reportDueDate',
 'reportMonth',
 'totalMonthlyExpense',
 'totalMonthlyIncome'}

In [26]:
file="prof_subcontractors.txt"
dfN=pd.read_csv(f"/home/joe/bic_etl/cdos/lobbyist/data_source/new_format/{file}",delimiter="\t",encoding="latin")


In [27]:
file="Sub-Contractor.txt"
dfO=pd.read_csv(f"/home/joe/bic_etl/cdos/lobbyist/data_source/{file}",delimiter="\t",encoding="latin")

In [28]:
dfO.head()
dfO.shape

(12026, 23)

In [29]:
dfN.shape

(510, 17)

In [30]:
display(dfO.head())

,lobbyistLastName,lobbyistFirstName,lobbyistAddress1,lobbyistAddress2,lobbyistCity,lobbyistState,lobbyistZip,primaryLobbyistID,annualLobbyistRegistrationID,incomeAmountReceivedamountPaidToSubcontractor,subcontractorAddressLine1,subcontractorAddressLine2,subcontractorCity,subcontractorState,subcontractorZip,incomeAmountReceived,amountPaidToSubcontractor,incomeReceiptDate,subcontractorLastName,subcontractorFirstName,reportMonth,fiscalYear,reportDueDate
0,Ackerman Information Corporation,MARGARET,2181 Willow Court,NaN,DENVER,CO,80238,19907000180,20017000561,COLORADO CORONERS ASSOCIATION,"C/O BRUCE ZOBEL, MOFFAT COUNTY CORONER",621 YAMPA AVENUE,CRAIG,CO,81625,1666.67,833.33,12/31/2001,BALCEROVICH,STEVE,December,2002,01/15/2002
1,MUNIZ,CARMELITA,"1410 GRANT STREET, SUITE B205",NaN,DENVER,CO,80203,19987000339,20017000625,COLORADO ASSOCIATION OF ALCOHOL AND DRUG SERVICE P,"1410 GRANT STREET, SUITE B205",NaN,DENVER,CO,80203,1500.00,1500.00,01/07/2002,LEOPOLDUS,ANDREA,December,2002,01/15/2002
2,MCEVOY,JEANNE,3561 W 112TH CIR,NaN,WESTMINSTER,CO,80031-7167,19897000253,20017000762,DRC,60 FRONTAGE ROAD,NaN,ANDOVER,MD,01810,4000.00,2000.00,01/15/2002,JENSEN,ANNMARIE,January,2002,02/15/2002
3,GUTHRIE,SPENCER,947 AZURE WAY,NaN,LOUISVILLE,CO,80027,20017000982,20017000982,GLAXOSMITHKLINE,RESEARCH TRIANGLE PARK,NaN,RALEIGH,NC,80202,2000.00,2000.00,12/01/2001,HOLDREN,STEVEN,December,2002,01/15/2002
4,GUTHRIE,SPENCER,947 AZURE WAY,NaN,LOUISVILLE,CO,80027,20017000982,20017000982,GLAXOSMITHKLINE,RESEARCH TRIANGLE PARK,NaN,RALEIGH,NC,80202,2000.00,2000.00,11/30/2001,HOLDREN,STEVEN,October,2002,11/15/2001


In [31]:
display(dfN.head())

,lobbyistName,lobbyistLastName,lobbyistFirstName,lobbyistFirmName,lobbyistZip,primaryLobbyistID,annualLobbyistRegistrationId,lobbyistClient,amountPaidtoSubcontractor,subcontractorPaidDate,subcontractorName,subcontractorZip,subcontractorPrimaryID,reportMonth,fiscalYear,reportDueDate,runDate
0,"Attwood, Amy (Attwood Public Affairs)",Attwood,Amy,Attwood Public Affairs,80123,20135000068,20235075561,Douglas County,"3,125.00",01/26/2024,"BALCEROVICH, STEVE",80203,20017000766,January,2023-2024,02/15/2024,02/28/2024
1,"Attwood, Amy (Attwood Public Affairs)",Attwood,Amy,Attwood Public Affairs,80123,20135000068,20235075561,Holland and Hart,"18,216.65",01/29/2024,"Mello, Jennifer Lakins (August Policy Strategies)",80237,20175036329,January,2023-2024,02/15/2024,02/28/2024
2,"Attwood, Amy (Attwood Public Affairs)",Attwood,Amy,Attwood Public Affairs,80123,20135000068,20235075561,Douglas County,"3,125.00",12/31/2023,"BALCEROVICH, STEVE",80203,20017000766,December,2023-2024,01/16/2024,02/28/2024
3,"Attwood, Amy (Attwood Public Affairs)",Attwood,Amy,Attwood Public Affairs,80123,20135000068,20235075561,Holland and Hart,"16,833.33",12/31/2023,"Mello, Jennifer Lakins (August Policy Strategies)",80237,20175036329,December,2023-2024,01/16/2024,02/28/2024
4,"Attwood, Amy (Attwood Public Affairs)",Attwood,Amy,Attwood Public Affairs,80123,20135000068,20235075561,Douglas County,"3,125.00",11/29/2023,"BALCEROVICH, STEVE",80203,20017000766,November,2023-2024,12/15/2023,02/28/2024


In [32]:
print(dfO.shape)
print(dfN.shape)

(12026, 23)
(510, 17)


In [33]:
set(dfO.columns)-set(dfN.columns)

{'amountPaidToSubcontractor',
 'annualLobbyistRegistrationID',
 'incomeAmountReceived',
 'incomeAmountReceivedamountPaidToSubcontractor',
 'incomeReceiptDate',
 'lobbyistAddress1',
 'lobbyistAddress2',
 'lobbyistCity',
 'lobbyistState',
 'subcontractorAddressLine1',
 'subcontractorAddressLine2',
 'subcontractorCity',
 'subcontractorFirstName',
 'subcontractorLastName',
 'subcontractorState'}

In [34]:
set(dfN.columns)-set(dfO.columns)

{'amountPaidtoSubcontractor',
 'annualLobbyistRegistrationId',
 'lobbyistClient',
 'lobbyistFirmName',
 'lobbyistName',
 'runDate',
 'subcontractorName',
 'subcontractorPaidDate',
 'subcontractorPrimaryID'}

In [11]:
dfsOld.keys()

dict_keys(['Client.txt', 'SummaryInfo.txt', 'Expenses.txt', 'Income.txt', 'Sub-Contractor.txt', 'EmployerBillPosition.txt'])

In [63]:
needs = {}
print(f"{'File':25s} {'# rows':6s} {'Start':4s} {'End':4s}")


for file,df in dfsOld.items():
#    print(file)
#    display(df.head())
    if "fiscalYear" in df.columns:
        mn = df['fiscalYear'].min()
        mx = df['fiscalYear'].max()
        start = str(int(mn))
        end =  str(int(mx))
    else:
        start = "----"
        end = "----"
   #     print(df.columns)
   #     display(df.head())
        
            
    print(f"{file:25s} {float(df.shape[0]):6.0f} {start:4s} {end:4s}")
        
#    print("-------")


File                      # rows Start End 
Client.txt                 87995 ---- ----
SummaryInfo.txt           196254 1995 2025
Expenses.txt               70124 1995 2024
Income.txt                 26572 ---- ----
Sub-Contractor.txt         12026 2002 2024
EmployerBillPosition.txt  335425 1995 2024


In [62]:
print(f"{'File':35s} {'# rows':6s} {'Start':9s} {'End':9s}")

for file,df in dfsNew.items():
#    print(file)
#    display(df.head())
    df.columns = df.columns.str.lower()
    if "fiscalyear" in df.columns:
        mn = df['fiscalyear'].min()
        mx = df['fiscalyear'].max()
        start = str(mn)
        end =  str(mx)
    # elif 'registrationenddate' in df.columns:
    #      start = df['registrationenddate'].argmin()
    #      end = df['registrationenddate'].argmax()
    else:
        start = "----"
        end = "----"
   #     print(df.columns)
   #     display(df.head(3))
        
            
    print(f"{file:35s} {float(df.shape[0]):6.0f} {start:9s} {end:9s}")
        
#    print("-------")

File                                # rows Start     End      
prof_bills.txt                       35570 2023-2024 2023-2024
prof_clients.txt                      5349 2023-2024 2023-2024
prof_disclosure_report_summary.txt    5081 2023-2024 2023-2024
prof_expenditures.txt                  329 2023-2024 2023-2024
prof_income.txt                       9299 2023-2024 2023-2024
prof_lobbyist_directory.txt           4967 ----      ----     
prof_subcontractors.txt                510 2023-2024 2023-2024
state_lobbyist_bills.txt             42872 ----      ----     
state_lobbyist_directory.txt           225 ----      ----     
state_lobbyist_funds_hours.txt        7993 ----      ----     
state_lobbyist_officials.txt          6219 ----      ----     


In [49]:
df.columns.str.lower()

Index(['lobbyistname', 'lobbyistlastname', 'lobbyistfirstname', 'stateagency', 'lobbyistzip', 'primarylobbyistid', 'annuallobbyistregistrationid', 'stateofficiallastname', 'stateofficialfirstname', 'stateofficialmiddlename', 'stateofficialtitle', 'stateofficialsection', 'reportmonth', 'reportduedate', 'calendaryear', 'rundate'], dtype='object')